In [1]:
import torch
import torch.nn as nn
import math

In [2]:
class InputEmbeddings(nn.Module):
    '''Each token is represented as a fixed-size, high-dimensional, vector (d=512). (p. 5 sec. 3.4)'''

    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):

        return self.embedding(x) * math.sqrt(self.d_model) # Multiplication by sqrt(d_model) following the original paper 

In [6]:
inputembedding = InputEmbeddings(512, 100)
x = torch.arange(0, 10).unsqueeze(0)
# print(x.shape)
embed = inputembedding.forward(x)
print(inputembedding.forward(x).shape)

torch.Size([1, 10, 512])


In [7]:
class PositionalEncoding(nn.Module):
    '''To each token, we add a learned position embedding of the same dimension. (p. 6 sec. 3.5)'''

    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)

        # Create a matrix of shape (seq_len, d_model)
        pe = torch.zeros(seq_len, d_model)
        # Createa a vector of shape (seq_len, 1)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1) # (seq_len, 1)
        i_tensor = torch.arange(0, d_model, 2)
        div_term = torch.exp(i_tensor.float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term) # Odd
        pe[:, 1::2] = torch.cos(position * div_term) # Even

        pe = pe.unsqueeze(0) # (1, seq_len, d_model)

        print(pe.shape)

        self.register_buffer('pe', pe)

    def forward(self, x):
        print(x.shape[1])
        print(self.pe[:, :x.shape[1], :].shape)
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)
        return self.dropout(x)

In [8]:
positionalencoding = PositionalEncoding(512, 10, 0.0)
print(positionalencoding.forward(embed))

torch.Size([1, 10, 512])
10
torch.Size([1, 10, 512])
tensor([[[-20.1230,  -6.9996, -30.8518,  ..., -13.2015,  24.5432,  32.6050],
         [ 11.1261, -29.7135,  25.7151,  ...,   7.7930,   0.8295,  46.1661],
         [ 30.4875, -12.5208,  -6.6358,  ..., -17.7200, -29.6016,  -7.0630],
         ...,
         [ -9.9409, -24.0871,  33.2429,  ...,  -9.2121,  22.4591, -23.9559],
         [-12.3938,  -3.5658, -46.6128,  ..., -12.3402,  16.7971,   3.1768],
         [  1.1470,   5.0489,  18.2740,  ..., -23.3754,  -1.6045, -25.1838]]],
       grad_fn=<AddBackward0>)
